# 5. Análisis de Ranking: Cambios en la Posición de las Causas de Muerte

## 5.1. Ranking de causas por año

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")


Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
# Tabla de cambios de ranking 1999 vs 2017
nac = estados[estados['cause_name']!='All causes'].groupby(['year','cause_name'])['deaths'].sum().reset_index()

def get_rank(year):
    d = nac[nac['year']==year].copy()
    d['rank'] = d['deaths'].rank(ascending=False, method='min').astype(int)
    return d[['cause_name','rank','deaths']].rename(columns={'rank':f'rank_{year}','deaths':f'muertes_{year}'})

r99 = get_rank(1999)
r17 = get_rank(2017)
merged = r99.merge(r17, on='cause_name')
merged['cambio'] = merged['rank_2017'] - merged['rank_1999']
merged['pct'] = (merged['muertes_2017'] - merged['muertes_1999']) / merged['muertes_1999'] * 100
merged = merged.sort_values('rank_1999')

def fmt_cambio(x):
    if x < 0: return f"▲ {abs(int(x))}"
    elif x > 0: return f"▼ {int(x)}"
    return "="

merged['Cambio'] = merged['cambio'].apply(fmt_cambio)
tabla = merged[['cause_name','rank_1999','rank_2017','Cambio','pct']].copy()
tabla.columns = ['Causa','Rank 1999','Rank 2017','Cambio','Variación %']
tabla['Variación %'] = tabla['Variación %'].round(1)
print(tabla.to_string(index=False))

                  Causa  Rank 1999  Rank 2017 Cambio  Variación %
          Heart disease          1          1      =        -10.7
                 Cancer          2          2      =          9.0
                 Stroke          3          5    ▼ 2        -12.5
                   CLRD          4          4      =         29.0
 Unintentional injuries          5          3    ▲ 2         73.7
               Diabetes          6          7    ▼ 1         22.2
Influenza and pneumonia          7          8    ▼ 1        -12.6
    Alzheimer's disease          8          6    ▲ 2        172.6
         Kidney disease          9          9      =         42.5
                Suicide         10         10      =         61.6


## 5.2. Cambios en el ranking: 1999 vs 2017

In [3]:
# Bump chart
fig = go.Figure()
colors = px.colors.qualitative.Set2
for i, row in merged.iterrows():
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=[1999, 2017],
        y=[row['rank_1999'], row['rank_2017']],
        mode='lines+markers+text',
        line=dict(color=color, width=2),
        marker=dict(size=10, color=color),
        text=[f"#{int(row['rank_1999'])} {row['cause_name'][:22]}",
              f"#{int(row['rank_2017'])} {row['cause_name'][:22]}"],
        textposition=['middle left','middle right'],
        textfont=dict(size=9),
        name=row['cause_name'][:22],
        showlegend=False
    ))
fig.update_layout(
    title='Cambios en el Ranking de Causas de Muerte (1999 vs 2017)',
    xaxis=dict(tickvals=[1999, 2017], tickfont=dict(size=13)),
    yaxis=dict(title='Ranking (1 = mayor tasa)', autorange='reversed'),
    height=580, template='plotly_white'
)
fig.show()

**Interpretación:** Las enfermedades cardíacas y el cáncer se mantienen como las dos primeras causas en todo el período. En contraste, las lesiones no intencionales suben del puesto 5 al 3 (+73.7%), y el Alzheimer asciende del 8 al 6 (+172.6%), reflejando el impacto del envejecimiento poblacional. El stroke baja del 3 al 5 (–12.5%).